# Unidad 45. EDA reproducible y comunicación

**Bloque:** Seaborn (UT3) · **Tiempo orientativo:** 5–7 horas

> Un análisis exploratorio reproducible conserva el camino desde datos locales hasta una conclusión limitada: inspección, transformación explícita, figura legible y exportación que otra persona pueda repetir.


## Resultados de aprendizaje

Al terminar podrás combinar filtrado, columnas derivadas, agrupaciones y tablas pivote de Pandas con figuras de Seaborn; elegir paletas y etiquetas accesibles; refinar una figura con Matplotlib; y exportarla en una ruta relativa.

**Prerequisitos:** unidades 18–23, 31–35 y 39–44.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

def ruta_datos(nombre):
    for ruta in [Path("datos") / "seaborn" / nombre, Path("..") / "datos" / "seaborn" / nombre]:
        if ruta.exists(): return ruta
    raise FileNotFoundError(f"No se encontró {nombre}. Abre el cuaderno desde la carpeta del curso.")

observaciones = pd.read_csv(ruta_datos("44_observaciones_modelo.csv"))
observaciones.info()


## 1. Un flujo que otra persona pueda revisar

Un EDA no es una colección de gráficos. Empieza con una pregunta, inspecciona las columnas y la calidad, transforma con operaciones visibles y produce figuras que respondan a esa pregunta. Evita modificar el `DataFrame` original sin dejar rastro.

Pregunta de ejemplo: **«¿Qué equilibrio aparece entre exactitud, latencia y energía en las observaciones simuladas?»**. Los datos son sintéticos; la respuesta solo describe esta tabla.


In [ ]:
assert observaciones.isna().sum().sum() == 0

analisis = observaciones.assign(
    eficiencia_pct_wh=observaciones["exactitud_pct"] / observaciones["energia_wh"],
    nivel_latencia=pd.cut(observaciones["latencia_ms"], bins=[0, 150, 200, 300], labels=["baja", "media", "alta"]),
)
analisis.head()


## 2. Resumir con Pandas antes de dibujar

Una agrupación responde a una pregunta de resumen. La siguiente tabla muestra medias por dispositivo y configuración; no sustituye a las observaciones individuales. Una tabla pivote deja visible qué filas, columnas, medida y agregación intervienen. Revisa ambas antes de interpretar barras o colores.


In [ ]:
resumen = analisis.groupby(["dispositivo", "configuracion"], as_index=False).agg(exactitud_media=("exactitud_pct", "mean"), latencia_media=("latencia_ms", "mean"), energia_media=("energia_wh", "mean"))
display(resumen)

pivote = analisis.pivot_table(index="configuracion", columns="dispositivo", values="eficiencia_pct_wh", aggfunc="mean", observed=False)
display(pivote)


## 3. Comunicar sin depender solo del color

Una paleta debe diferenciar grupos, pero el color no debe ser la única señal. Aquí `hue` se refuerza con `style`, las etiquetas incluyen unidades y el título describe exactamente la relación. Matplotlib permite añadir ajustes razonados y exportar la figura.

No recortes el eje vertical para exagerar diferencias pequeñas. Una anotación debe señalar una observación o decisión que ayude a leer, no decorar el gráfico.


In [ ]:
paleta = {"base": "#4477AA", "ajustada": "#CC6677"}
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=analisis, x="energia_wh", y="exactitud_pct", hue="configuracion", style="configuracion", palette=paleta, s=80, ax=ax)
ax.set(title="Exactitud y energía en observaciones simuladas", xlabel="Energía (Wh)", ylabel="Exactitud (%)")
ax.legend(title="Configuración")
ax.annotate("Dispositivo edge", xy=(12.4, 70), xytext=(15.5, 69), arrowprops={"arrowstyle": "->"})
fig.tight_layout()
plt.show()


## 4. Exportar sin rutas absolutas

Una ruta absoluta solo funciona en el equipo de quien la escribió. Usa `Path` y una carpeta relativa al curso. La celda siguiente está comentada para no generar archivos al ejecutar el cuaderno de ejemplo. Al entregar una actividad, comprueba que la figura se exportó y que el archivo generado no se añade al repositorio salvo que se solicite.


In [ ]:
# salida = Path("unidades") / "salidas" / "45_exactitud_energia.png"
# salida.parent.mkdir(parents=True, exist_ok=True)
# fig.savefig(salida, dpi=150, bbox_inches="tight")


## Práctica guiada

Filtra solo `dispositivo == "servidor"`, crea una columna `latencia_por_mb` mediante una operación vectorizada y construye una dispersión de memoria frente a latencia con configuración como color y estilo. Indica en una frase que la figura describe únicamente ese subconjunto.


In [ ]:
# Completa la práctica guiada aquí.
servidor = (
    observaciones[observaciones["dispositivo"] == "servidor"]
    .copy()
    .assign(latencia_por_mb=lambda df: df["latencia_ms"] / df["memoria_mb"])
)

fig, ax = plt.subplots(figsize=(7.2, 4.3))
sns.scatterplot(
    data=servidor,
    x="memoria_mb",
    y="latencia_ms",
    hue="configuracion",
    style="configuracion",
    s=85,
    ax=ax,
)
ax.set(
    title="Latencia frente a memoria en el subconjunto 'servidor'",
    xlabel="Memoria (MB)",
    ylabel="Latencia (ms)",
)
ax.legend(title="Configuración")
plt.show()

# Esta figura describe únicamente las observaciones simuladas del subconjunto filtrado
# correspondiente al dispositivo 'servidor' y no debe generalizarse al dispositivo 'edge'.

- **[U45.1]** `servidor = observaciones[observaciones["dispositivo"] == "servidor"].copy().assign(latencia_por_mb=lambda df: df["latencia_ms"] / df["memoria_mb"])` filtra el subconjunto `servidor` sin alterar `observaciones` y calcula de forma vectorizada la nueva columna `latencia_por_mb`,
- **[U45.3]** `sns.scatterplot(data=servidor, x="memoria_mb", y="latencia_ms", hue="configuracion", style="configuracion", s=85, ax=ax)` dibuja la dispersión diferenciando cada configuración por color y forma del marcador[cite: 8],
- **[U45.3]** `ax.set(...)`, `ax.legend(title="Configuración")` y `plt.show()` titulan la figura indicando explícitamente el subconjunto representado e incluyen unidades (`MB` y `ms`)[cite: 8].

## Ejercicios

### Ejercicio 45.01 · Transformación trazable

Crea una copia llamada `edge` con las observaciones del dispositivo edge. Añade `eficiencia_pct_wh` y verifica que tiene 12 filas y no contiene valores ausentes.

### Ejercicio 45.02 · Resumen y figura coherentes

Agrupa por configuración y calcula medias de exactitud y energía. Representa esas medias con un gráfico que indique explícitamente que son medias simuladas.

### Ejercicio 45.03 · Pivote y mapa de calor

Crea una tabla pivote de latencia media por configuración y dispositivo. Represéntala con un `heatmap` con anotaciones, barra de color y unidades.

### Ejercicio 45.04 · Diseño accesible

Crea una dispersión de latencia frente a exactitud. Usa color y estilo para configuración, título, etiquetas y leyenda. Escribe por qué una persona no debe depender solo del color para interpretar grupos.

### Ejercicio 45.05 · Miniproyecto: mini EDA reproducible

Entrega una secuencia de cuatro pasos: inspección de calidad, transformación documentada, una tabla de resumen y dos figuras complementarias. Una figura debe mostrar puntos individuales; la otra puede resumir con paneles o un mapa de calor. Redacta conclusiones, límites y la ruta relativa que usarías para exportar una figura.


In [ ]:
# Escribe aquí tu solución de los ejercicios 45.01 a 45.05.

# ==========================================
# Ejercicio 45.01 · Transformación trazable
# ==========================================
edge = observaciones[observaciones["dispositivo"] == "edge"].copy()
assert (edge["energia_wh"] > 0).all(), "Existen registros con energia_wh <= 0"

edge = edge.assign(eficiencia_pct_wh=edge["exactitud_pct"] / edge["energia_wh"])

assert len(edge) == 12, f"Se esperaban 12 filas en edge, pero hay {len(edge)}"
assert not edge.isna().any().any(), "Existen valores ausentes en edge"
display(edge.head())

# ==========================================
# Ejercicio 45.02 · Resumen y figura coherentes
# ==========================================
resumen_config = (
    observaciones.groupby("configuracion", as_index=False)
    .agg(
        exactitud_media=("exactitud_pct", "mean"),
        energia_media=("energia_wh", "mean"),
    )
    .round(2)
)
display(resumen_config)

fig, ejes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
sns.barplot(
    data=resumen_config,
    x="configuracion",
    y="exactitud_media",
    hue="configuracion",
    legend=False,
    ax=ejes[0],
)
ejes[0].set(
    title="Exactitud media simulada",
    xlabel="Configuración",
    ylabel="Exactitud media (%)",
)

sns.barplot(
    data=resumen_config,
    x="configuracion",
    y="energia_media",
    hue="configuracion",
    legend=False,
    ax=ejes[1],
)
ejes[1].set(
    title="Energía media simulada",
    xlabel="Configuración",
    ylabel="Energía media (Wh)",
)
fig.suptitle("45.02 · Medias simuladas de exactitud y consumo energético por configuración")
plt.show()

# ==========================================
# Ejercicio 45.03 · Pivote y mapa de calor
# ==========================================
pivote_latencia = observaciones.pivot_table(
    index="configuracion",
    columns="dispositivo",
    values="latencia_ms",
    aggfunc="mean",
    observed=False,
)
display(pivote_latencia.round(2))

fig, ax = plt.subplots(figsize=(6.2, 3.8))
sns.heatmap(
    pivote_latencia,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    linewidths=0.5,
    cbar_kws={"label": "Latencia media (ms)"},
    ax=ax,
)
ax.set(
    title="45.03 · Latencia media simulada (ms) por configuración y dispositivo",
    xlabel="Dispositivo",
    ylabel="Configuración",
)
plt.show()

# ==========================================
# Ejercicio 45.04 · Diseño accesible
# ==========================================
paleta_accesible = {"base": "#4477AA", "ajustada": "#CC6677"}

fig, ax = plt.subplots(figsize=(7.2, 4.3))
sns.scatterplot(
    data=observaciones,
    x="latencia_ms",
    y="exactitud_pct",
    hue="configuracion",
    style="configuracion",
    markers={"base": "o", "ajustada": "s"},
    palette=paleta_accesible,
    s=85,
    ax=ax,
)
ax.set(
    title="45.04 · Exactitud frente a latencia por configuración",
    xlabel="Latencia (ms)",
    ylabel="Exactitud (%)",
)
ax.legend(title="Configuración")
plt.show()
# Explicación: No debe dependerse solo del color porque personas con daltonismo o deficiencias en
# la percepción cromática (o al imprimir la figura en escala de grises) no podrían distinguir los
# grupos si no van reforzados por una segunda señal visual como la forma del marcador (style).

# ==========================================
# Ejercicio 45.05 · Miniproyecto: mini EDA reproducible
# ==========================================
# Paso 1: Inspección de calidad
assert observaciones.shape == (24, 6), f"Dimensión inesperada: {observaciones.shape}"
assert not observaciones.isna().any().any(), "Existen valores ausentes"
assert (observaciones["energia_wh"] > 0).all(), "Energía no positiva detectada"

# Paso 2: Transformación documentada sin mutar el DataFrame original
eda_df = observaciones.copy().assign(
    eficiencia_pct_wh=observaciones["exactitud_pct"] / observaciones["energia_wh"]
)

# Paso 3: Tabla de resumen agrupada
tabla_eda = (
    eda_df.groupby(["dispositivo", "configuracion"], as_index=False)
    .agg(
        exactitud_media=("exactitud_pct", "mean"),
        latencia_media=("latencia_ms", "mean"),
        energia_media=("energia_wh", "mean"),
        eficiencia_media=("eficiencia_pct_wh", "mean"),
    )
    .round(2)
)
display(tabla_eda)

# Paso 4: Dos figuras complementarias (puntos individuales + resumen matricial)
fig, ejes = plt.subplots(1, 2, figsize=(12.5, 4.5), layout="constrained")

sns.scatterplot(
    data=eda_df,
    x="latencia_ms",
    y="exactitud_pct",
    hue="configuracion",
    style="dispositivo",
    palette=paleta_accesible,
    s=85,
    ax=ejes[0],
)
ejes[0].set(
    title="1. Observaciones individuales (exactitud vs. latencia)",
    xlabel="Latencia (ms)",
    ylabel="Exactitud (%)",
)
ejes[0].legend(title="Configuración / Dispositivo")

pivote_eficiencia = eda_df.pivot_table(
    index="configuracion",
    columns="dispositivo",
    values="eficiencia_pct_wh",
    aggfunc="mean",
    observed=False,
)
sns.heatmap(
    pivote_eficiencia,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    linewidths=0.5,
    cbar_kws={"label": "Eficiencia media (%/Wh)"},
    ax=ejes[1],
)
ejes[1].set(
    title="2. Eficiencia energética media simulada (%/Wh)",
    xlabel="Dispositivo",
    ylabel="Configuración",
)

fig.suptitle("45.05 · Mini EDA reproducible: rendimiento y eficiencia simulada")
plt.show()

# Ruta relativa portable recomendada para exportar (comentada para no ensuciar el repositorio):
# ruta_exportacion = Path("unidades") / "salidas" / "45_mini_eda_reproducible.png"
# ruta_exportacion.parent.mkdir(parents=True, exist_ok=True)
# fig.savefig(ruta_exportacion, dpi=150, bbox_inches="tight")


### Explicación línea por línea (Ejercicios 45.01 a 45.05)

- **[U45.1]** `edge = observaciones[observaciones["dispositivo"] == "edge"].copy()` crea una copia independiente del subconjunto `edge` para no mutar `observaciones`, y `.assign(eficiencia_pct_wh=edge["exactitud_pct"] / edge["energia_wh"])` añade la métrica derivada tras comprobar que `energia_wh > 0`[cite: 8],
- **[U45.2]** `resumen_config = observaciones.groupby("configuracion", as_index=False).agg(...).round(2)` calcula las medias por configuración antes de representarlas en gráficos de barras etiquetados explícitamente como medias simuladas[cite: 8],
- **[U45.2]** `pivote_latencia = observaciones.pivot_table(index="configuracion", columns="dispositivo", values="latencia_ms", aggfunc="mean", observed=False)` construye la matriz de latencias medias para dibujarla con `sns.heatmap(..., annot=True, fmt=".1f")`[cite: 8],
- **[U45.3]** `sns.scatterplot(..., hue="configuracion", style="configuracion", markers={"base": "o", "ajustada": "s"}, palette=paleta_accesible, ...)` combina paleta accesible y marcadores distintos para que los grupos sean legibles sin depender exclusivamente del color[cite: 8],
- **[U45.1 / U45.4]** El Ejercicio 45.05 encadena las cuatro fases reproducibles: aserciones de calidad, transformación trazable con `.copy().assign()`, tabla de resumen agrupada, figura doble (puntos individuales + mapa de calor de eficiencia) y ruta relativa portable con `Path("unidades") / "salidas" / "45_mini_eda_reproducible.png"`[cite: 8].

---

### Conclusiones del Ejercicio 45.05
1. **Conclusión principal:** En estas 24 observaciones simuladas, el dispositivo `edge` alcanza mayor eficiencia energética relativa (`%/Wh`) debido a su bajo consumo en vatios-hora a costa de mayores latencias, mientras que `servidor` logra menor latencia con mayor consumo energético y la configuración `ajustada` mejora la exactitud media en ambos entornos[cite: 8].
2. **Límites:** Los resultados describen exclusivamente un conjunto de datos sintéticos pequeño (6 observaciones por combinación) sin repeticiones ni contexto de carga real, por lo que no permiten establecer causalidad ni garantizar el mismo rendimiento en producción[cite: 8].
3. **Ruta relativa de exportación:** Para guardar la figura de forma portable entre equipos se emplearía `Path("unidades") / "salidas" / "45_mini_eda_reproducible.png"` junto con `mkdir(parents=True, exist_ok=True)` y `bbox_inches="tight"`[cite: 8].

## Autoevaluación y cierre

1. ¿Por qué separar el `DataFrame` original del transformado?
2. ¿Qué debe indicar el título de una figura basada en medias?
3. ¿Qué información debe conservar una figura accesible además del color?
4. ¿Por qué exportar con una ruta relativa?
5. ¿Qué componentes hacen reproducible un EDA?

Has completado la preparación para el proyecto 46, donde aplicarás el flujo completo sobre un conjunto de datos documentado.
